# **Assignment 1 - Crew Scheduling Problem**


```
The format of these data files is:  
   number of rows, number of columns (n)  
   for each column j (j=1,...,n) in turn:  
      column cost, number of rows covered by j, list of the rows covered by j  
```

### **Problem Statement:**
 + Select a subset of columns to minimize cost while following the constraint that each row must be covered EXACTLY ONCE by the subset.
 + From googling, it looks like the true optimals are: `sppnw41: 11307` `sppnw42: 7656` `sppnw43: 8904`

In [1]:
# Variable Setup

import numpy as np
f_name ='sppnw42'
file_path = f'data/{f_name}.txt'
with open(file_path, 'r') as f:
    data = f.readlines()

if f_name == 'sppnw41': optimal = 11307
elif f_name == 'sppnw42': optimal = 7676
elif f_name == 'sppnw43': optimal = 8904
else: optimal = 0

data = [[int(x) for x in d.strip().split()] for d in data]

N_ROWS,N_COLS = data[0]

COL_COSTS = np.array([d[0] for d in data[1:]])
COL_ROWS = np.array([np.isin(np.arange(1,N_ROWS+1), d[2:]) for d in data[1:]])

print(f"Num Rows: {N_ROWS} | Num Columns: {N_COLS}")
print(f"Avg Rows per Column: {np.mean([len(rows[rows]) for rows in COL_ROWS]):.2f}")

Num Rows: 23 | Num Columns: 1079
Avg Rows per Column: 6.05


In [2]:
# Getting some idea of scale

total_cost = np.sum(COL_COSTS); avg_cost = np.mean(COL_COSTS)
min_cost = np.min(COL_COSTS); max_cost = np.max(COL_COSTS)
std_cost = np.std(COL_COSTS)

print(f"Total cost: {total_cost} | Avg Cost: {avg_cost:.0f} | STD of Cost: {std_cost:.0f}")
print(f"Min Cost: {min_cost} | Max Cost: {max_cost}")

Total cost: 3299180 | Avg Cost: 3058 | STD of Cost: 1198
Min Cost: 140 | Max Cost: 6516


In [ ]:
# Binary Genetic Algorithm (BGA)


# ---- Binary Representation

#   Individuals are defined by a binary string COLUMNS long
INDIVIDUAL_SHAPE = (N_COLS)

POPULATION_SIZE = 2048

NUM_PARENTS = int(POPULATION_SIZE * 1/8)
NUM_CHILDREN = 2048
NUM_ELITES = int(POPULATION_SIZE * 0.01)

POPULATION_SHAPE = (POPULATION_SIZE, INDIVIDUAL_SHAPE)
# --------------


#   Mutation and Initialization Parameters
#       - This is because the files have a massively varying number of columns
MUTATION_RATE = 4.0 / N_COLS #N_COLS # want around 2-6 columns changed per mutation
INIT_SELECT_RATE = 10.0 / N_COLS # only want to start with around 10 columns selected, otherwise we get stuck in massively costly and unfit solutions


# ---- FITNESS FUNCTIONS - Same as taken from SA
# Fitness Functions
def cost(solution: np.ndarray) -> int:
    return np.sum(COL_COSTS[solution])

def penalty(solution: np.ndarray, zero_w: int, overlap_w: int) -> int:
    cols = COL_ROWS[solution]
    row_sums = np.sum(cols, axis=0)

    zero_rows = row_sums[row_sums < 1]
    overlap_rows = row_sums[row_sums > 1]

    return zero_rows.size * zero_w + np.sum(overlap_rows - 1) * overlap_w

def fitness(solution: np.ndarray, zero_w: int, overlap_w: int) -> int:
    return cost(solution) + penalty(solution,zero_w=zero_w,overlap_w=overlap_w)
# --------------------


def pop_fitness(p: np.ndarray, zero_weight, overlap_weight) -> tuple[np.ndarray, np.ndarray]:
    row_sums = p.astype(int) @ COL_ROWS
    costs = p @ COL_COSTS

    zero_pen = (row_sums < 1).sum(axis=1)
    overlap_pen = np.maximum(row_sums - 1, 0).sum(axis=1)

    fitnesses = costs + zero_weight * zero_pen + overlap_weight * overlap_pen
    feas = (zero_pen + overlap_pen) == 0

    return fitnesses, feas
# --------------------

def selection(p, fitness, feasibility, num_parents):
    N = p.shape[0]
    parents = []

    for _ in range(num_parents):
        i, j = np.random.randint(0, N, 2)

        if feasibility[i] and not feasibility[j]:
            winner = i
        elif feasibility[j] and not feasibility[i]:
            winner = j
        else:  
            winner = i if fitness[i] < fitness[j] else j
        
        parents.append(p[winner])

    return np.array(parents)

def mutation(p, pm): # based on   "Lecture5_EvolutionaryAlgorithms-6.pdf" Slide 20/25, using fixed pm
    bit_flips = np.random.rand(*p.shape) < pm
    p[bit_flips] = ~p[bit_flips]
    return p

def crossover(parents, num_children): # based on   "Lecture5_EvolutionaryAlgorithms-6.pdf" Slide 21/25, using uniform crossover
    children = []

    def parent_crossover(x1, x2):
        mask = np.random.rand(INDIVIDUAL_SHAPE) < 0.5
        c1 = x1.copy()
        c2 = x2.copy()
        c1[mask] = x2[mask]
        c2[mask] = x1[mask]
        return c1, c2

    while len(children) < num_children:
        i, j = np.random.choice(len(parents), 2, replace=False)
        c1, c2 = parent_crossover(parents[i], parents[j])

        children.append(c1)

        if len(children) < num_children:
            children.append(c2)

    return np.array(children)



def diversity_hamming_mean(p: np.ndarray) -> float:
    # p: (pop, n) bool
    x = p.astype(np.int8, copy=False)
    pop, n = x.shape
    ones = x.sum(axis=0)              # (n,)
    zeros = pop - ones
    # expected Hamming distance between two random individuals:
    # for each bit: P(diff)=2p(1-p) = 2*(ones/pop)*(zeros/pop)
    return float((2 * ones * zeros).sum() / (pop * (pop - 1)))



#
def binary_genetic_algorithm(p0: np.ndarray|None=None, max_iter=10000):
    if p0 is not None and p0.shape != POPULATION_SHAPE:
        raise ValueError("Initialisation population shape is invalid.") 
    
    #zero_weight_range = [1000,10000]; overlap_weight_range = [1000,10000]
    #def weights(gen):
    #    zero_w = zero_weight_range[0] + (gen/max_iter) * (zero_weight_range[1] - zero_weight_range[0])
    #    overlap_w = overlap_weight_range[0] + (gen/max_iter) * (overlap_weight_range[1] - overlap_weight_range[0])
    #    return zero_w,overlap_w
    #
    zero_w,overlap_w = 3000, 2000

    p = p0 if p0 is not None else np.random.rand(*POPULATION_SHAPE) < INIT_SELECT_RATE
    p_fit, p_feas = pop_fitness(p, zero_weight=zero_w, overlap_weight=overlap_w)

    gen = 0

    while gen < max_iter:
        #zero_w,overlap_w = weights(gen)

        p_parent = selection(p, p_fit, p_feas, NUM_PARENTS)
        p_new = crossover(p_parent, NUM_CHILDREN)
        p_new = mutation(p_new, pm=MUTATION_RATE) 
        p_new_fit, p_new_feas = pop_fitness(p_new, zero_weight=zero_w, overlap_weight=overlap_w)



        # Replacement (in p_fit and p) using elitism.
        p_combined = np.vstack([p,p_new])
        p_fit_combined = np.concatenate([p_fit,p_new_fit])
        p_feas_combined = np.concatenate([p_feas,p_new_feas])


        rank_idx = np.lexsort((p_fit_combined,~p_feas_combined)) # Best are feasible, next best have good fitness
        elite_idx = np.lexsort((p_fit,~p_feas))[:NUM_ELITES]

        remaining = POPULATION_SIZE - NUM_ELITES
        rank_idx = rank_idx[~np.isin(rank_idx,elite_idx)] # works because p is before p_new
        selected_idx = rank_idx[:remaining]

        

        p = np.vstack([p[elite_idx], p_combined[selected_idx]])
        p_fit = np.concatenate([p_fit[elite_idx], p_fit_combined[selected_idx]])
        p_feas = np.concatenate([p_feas[elite_idx], p_feas_combined[selected_idx]])
        # ----


        # Tracking best & logging
        best_i = np.argmin(p_fit)
        best = p[best_i]; fit_best = p_fit[best_i]

        if gen % 20 == 0:
            c = cost(best); f, _ = fitness(best, zero_weight=zero_w, overlap_weight=overlap_w)
            feas_percent = len([p for p in p_feas if p])/len(p_feas)
            diversity = diversity_hamming_mean(p)
            pop_child_proportion = (len(selected_idx[selected_idx >= len(p_fit)]) / (len(selected_idx) + len(elite_idx)))
            avg_fit = np.mean(p_fit)
            print(f"Cost: {c:.0f} | Fit: {fit_best:.0f} | {'X' if c != fit_best else 'Y'} | Gen: {gen} | Feasible %: {feas_percent:.3f} | Diversity: {diversity:.2f} | Pop Child Proportion: {pop_child_proportion:.3f} | Avg Fit: {avg_fit:.0f}")#Penalty Weights: {zero_w:.0f},{overlap_w:.0f} ")
        # ----
        
        gen += 1

        if fit_best <= optimal:
            return best


    if np.any(p_feas):
        feas_idx = np.where(p_feas)[0]
        final_i = feas_idx[np.argmin(p_fit[feas_idx])]
    else:
        final_i = np.argmin(p_fit)
    return p[final_i]

#p0 = np.random.randint(0,2,N_COLS,dtype=bool)
best = binary_genetic_algorithm(max_iter=10000)
best_cols = COL_ROWS[best]
row_sums = np.sum(best_cols, axis=0)

print(f"\nFinal number of columns: {best[best].size} | Overlaps: {row_sums[row_sums > 1].size} | Uncovered: {row_sums[row_sums < 1].size}")
print(row_sums)
print(f"Final Cost: {cost(best)} | Feasible: {'YES' if np.all(row_sums == 1) else 'NO'} | Columns used:", np.where(best)[0])


Cost: 12702 | Fit: 29702 | X | Gen: 0 | Feasible %: 0.000 | Diversity: 17.07 | Pop Child Proportion: 0.373 | Avg Fit: 103156
Cost: 6762 | Fit: 22762 | X | Gen: 20 | Feasible %: 0.000 | Diversity: 9.15 | Pop Child Proportion: 0.086 | Avg Fit: 48431
Cost: 6044 | Fit: 20044 | X | Gen: 40 | Feasible %: 0.000 | Diversity: 8.40 | Pop Child Proportion: 0.046 | Avg Fit: 40947
Cost: 6044 | Fit: 20044 | X | Gen: 60 | Feasible %: 0.000 | Diversity: 8.20 | Pop Child Proportion: 0.030 | Avg Fit: 37327
Cost: 9482 | Fit: 15482 | X | Gen: 80 | Feasible %: 0.000 | Diversity: 8.21 | Pop Child Proportion: 0.016 | Avg Fit: 35029
Cost: 9482 | Fit: 15482 | X | Gen: 100 | Feasible %: 0.000 | Diversity: 8.14 | Pop Child Proportion: 0.015 | Avg Fit: 33222


KeyboardInterrupt: 